# Assignment 4: R-to-Python Parity

Question: how do mean BMI, sample SD, and observation counts vary by income group and gender among adults aged 20-80? The comparison uses the same cached CSV and the same complete-case and age filters in both languages.

In [1]:
from pathlib import Path
import sys
import pandas as pd

data_relative_path = Path("examples/nhanes-equity/data/nhanes_equity_v6.csv")
workspace_root = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / data_relative_path).exists()),
    None,
)
if workspace_root is None:
    raise FileNotFoundError(f"Could not locate {data_relative_path} from {Path.cwd()}")

submission_dir = workspace_root / "assignments/assignment04-polyglot/submission"
sys.path.insert(0, str(submission_dir))
nhanes = pd.read_csv(workspace_root / data_relative_path, low_memory=False)
nhanes.shape

(105626, 18)

In [2]:
raw_check = {
    "rows": len(nhanes),
    "columns": nhanes.shape[1],
    "missing_BMI": int(nhanes["BMI"].isna().sum()),
}
raw_check

{'rows': 105626, 'columns': 18, 'missing_BMI': 9356}

## Translation checks

- `is.na()` becomes `.isna()`; filter missing values explicitly before summarizing.
- R `sd()` uses sample SD; pandas `.std()` also defaults to `ddof=1`.
- R `group_by()` plus `summarise()` corresponds to pandas `groupby()` plus `agg()`.
- Keep the age bounds inclusive in both languages and round only after computing the summaries.

In [3]:
analysis = nhanes.loc[
    nhanes["BMI"].notna()
    & nhanes["IncomeGroup"].notna()
    & nhanes["Gender"].notna()
    & nhanes["Age"].between(20, 80, inclusive="both")
].copy()
filtered_row_count = len(analysis)
filtered_row_count

50851

In [4]:
summary = (
    analysis.groupby(["IncomeGroup", "Gender"], as_index=False, sort=True)
    .agg(
        n=("BMI", "size"),
        mean_bmi=("BMI", "mean"),
        sd_bmi=("BMI", "std"),
    )
)
summary[["mean_bmi", "sd_bmi"]] = summary[["mean_bmi", "sd_bmi"]].round(1)
summary

,IncomeGroup,Gender,n,mean_bmi,sd_bmi
0,High Income (>3.5),Female,8073,28.3,7.0
1,High Income (>3.5),Male,8257,28.8,5.6
2,Low Income (<1.3),Female,8471,30.5,8.1
3,Low Income (<1.3),Male,6822,28.1,6.4
4,Middle Income,Female,9954,29.8,7.5
5,Middle Income,Male,9274,28.8,6.2


## Reusable helper and parity assertions

The helper is kept in `helpers.py` so the notebook and future scripts can reuse the same filtering and summary logic.

In [6]:
from helpers import summarize_bmi_by_group

expected_r_summary = pd.DataFrame(
    {
        "IncomeGroup": [
            "High Income (>3.5)",
            "High Income (>3.5)",
            "Low Income (<1.3)",
            "Low Income (<1.3)",
            "Middle Income",
            "Middle Income",
        ],
        "Gender": ["Female", "Male", "Female", "Male", "Female", "Male"],
        "n": [8073, 8257, 8471, 6822, 9954, 9274],
        "mean_bmi": [28.3, 28.8, 30.5, 28.1, 29.8, 28.8],
        "sd_bmi": [7.0, 5.6, 8.1, 6.4, 7.5, 6.2],
    }
)
summary_from_helper = summarize_bmi_by_group(nhanes)
pd.testing.assert_frame_equal(summary, summary_from_helper)
pd.testing.assert_frame_equal(summary, expected_r_summary)
assert nhanes.shape == (105626, 18)
assert raw_check["missing_BMI"] == 9356
assert filtered_row_count == 50851
assert len(summary) == 6

parity_metrics = pd.DataFrame(
    {
        "metric": ["raw_row_count", "raw_column_count", "missing_BMI", "filtered_row_count", "number_of_groups"],
        "python_value": [len(nhanes), nhanes.shape[1], raw_check["missing_BMI"], filtered_row_count, len(summary)],
    }
)
parity_metrics

,metric,python_value
0,raw_row_count,105626
1,raw_column_count,18
2,missing_BMI,9356
3,filtered_row_count,50851
4,number_of_groups,6


In [ ]:
summary

## AI-use note

AI assisted with drafting and comparing the translation. I ran the notebook and checked the raw dimensions, missing BMI count, filter count, group labels, and rounded values against the R report. The pandas standard-deviation setting was checked against R's sample SD convention.